In [1]:
import os

# Fix TF/protobuf crash reliably in this environment by forcing pure-Python protobuf
# and ensuring it's set BEFORE importing tensorflow.
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "2"

import re
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)



2026-01-22 11:56:10.583636: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769082970.597954      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769082970.602741      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-22 11:56:10.623457: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
TRAIN_IMG_DIR = "/kaggle/input/petfinder-pawpularity-score/train/"
TEST_IMG_DIR = "/kaggle/input/petfinder-pawpularity-score/test/"
TRAIN_CSV_PATH = "/kaggle/input/petfinder-pawpularity-score/train.csv"
TEST_CSV_PATH = "/kaggle/input/petfinder-pawpularity-score/test.csv"
SAMPLE_SUB_PATH = "/kaggle/input/petfinder-pawpularity-score/sample_submission.csv"

relevant_columns = [
    "Subject Focus",
    "Eyes",
    "Face",
    "Near",
    "Action",
    "Accessory",
    "Group",
    "Collage",
    "Human",
    "Occlusion",
    "Info",
    "Blur",
]
target_col = "Pawpularity"

train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

train_feat_map = {
    row["Id"]: row[relevant_columns].to_numpy(dtype=np.float32)
    for _, row in train_df.iterrows()
}
train_y_map = {row["Id"]: np.float32(row[target_col]) for _, row in train_df.iterrows()}

test_feat_map = {
    row["Id"]: row[relevant_columns].to_numpy(dtype=np.float32)
    for _, row in test_df.iterrows()
}

all_train_ids = train_df["Id"].tolist()
rng = np.random.RandomState(SEED)
rng.shuffle(all_train_ids)
split_idx = int(len(all_train_ids) * 0.95)
train_ids = all_train_ids[:split_idx]
valid_ids = all_train_ids[split_idx:]

len(train_ids), len(valid_ids)



(8474, 446)

In [3]:
IMG_SIZE = (256, 256)


def _load_image_np(img_path: str) -> np.ndarray:
    img = tf.keras.utils.load_img(img_path, color_mode="rgb", target_size=IMG_SIZE)
    img = np.array(
        img, dtype=np.float32
    )  # [0..255], no normalization to preserve semantics
    return img


def pythonic_loader_train():
    for pid in train_ids:
        img = _load_image_np(os.path.join(TRAIN_IMG_DIR, f"{pid}.jpg"))
        features = train_feat_map[pid]
        y = train_y_map[pid]
        yield ({"Image": img, "Feature": features}, y)


def pythonic_loader_valid():
    for pid in valid_ids:
        img = _load_image_np(os.path.join(TRAIN_IMG_DIR, f"{pid}.jpg"))
        features = train_feat_map[pid]
        y = train_y_map[pid]
        yield ({"Image": img, "Feature": features}, y)




In [4]:
train_loader = tf.data.Dataset.from_generator(
    pythonic_loader_train,
    output_signature=(
        {
            "Image": tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
            "Feature": tf.TensorSpec(shape=(12,), dtype=tf.float32),
        },
        tf.TensorSpec(shape=(), dtype=tf.float32),
    ),
)

valid_loader = tf.data.Dataset.from_generator(
    pythonic_loader_valid,
    output_signature=(
        {
            "Image": tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
            "Feature": tf.TensorSpec(shape=(12,), dtype=tf.float32),
        },
        tf.TensorSpec(shape=(), dtype=tf.float32),
    ),
)



2026-01-22 11:56:19.377607: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1769082979.378260      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


In [5]:
# Fix Keras metrics/loss IndexError by using a single-tensor output (not a dict output).
# This keeps the architecture identical and avoids compile_utils expecting lists for multi-output.
input_image = tf.keras.Input(shape=(256, 256, 3), name="Image")
input_meta = tf.keras.Input(shape=(12,), name="Feature")  # shape must be tuple

l2 = tf.keras.layers.MaxPool2D((2, 2))(input_image)
l3 = tf.keras.layers.Conv2D(8, (3, 3), activation="relu")(l2)
l4 = tf.keras.layers.MaxPool2D((2, 2))(l3)
l5 = tf.keras.layers.Conv2D(16, (3, 3), activation="relu")(l4)
l6 = tf.keras.layers.MaxPool2D((2, 2))(l5)
l7 = tf.keras.layers.Conv2D(32, (3, 3), activation="relu")(l6)
l_mid1 = tf.keras.layers.MaxPool2D((2, 2))(l7)
l_mid2 = tf.keras.layers.Conv2D(64, (3, 3), activation="relu")(l_mid1)
l8 = tf.keras.layers.Flatten()(l_mid2)
l10 = tf.keras.layers.Dense(1024, activation="gelu")(l8)

combined = tf.keras.layers.concatenate([l10, input_meta])
l12 = tf.keras.layers.Dense(512, activation="gelu")(combined)
bn1 = tf.keras.layers.BatchNormalization()(l12)
le1 = tf.keras.layers.Dense(512, activation="gelu")(bn1)
bn2 = tf.keras.layers.BatchNormalization()(le1)
le2 = tf.keras.layers.Dense(512, activation="gelu")(bn2)
bn3 = tf.keras.layers.BatchNormalization()(le2)
le3 = tf.keras.layers.Dense(512, activation="gelu")(bn3)
bn4 = tf.keras.layers.BatchNormalization()(le3)
le4 = tf.keras.layers.Dense(256, activation="gelu")(bn4)
bn_5 = tf.keras.layers.BatchNormalization()(le4)
le5 = tf.keras.layers.Dense(256, activation="gelu")(bn_5)
bn_6 = tf.keras.layers.BatchNormalization()(le5)

le6 = tf.keras.layers.Dense(128, activation="gelu")(bn_6)
bn5 = tf.keras.layers.BatchNormalization()(le6)
le9 = tf.keras.layers.Dense(64, activation="gelu")(bn5)
bn6 = tf.keras.layers.BatchNormalization()(le9)
l13 = tf.keras.layers.Dense(16, activation="gelu")(bn6)
bn7 = tf.keras.layers.BatchNormalization()(l13)
l14 = tf.keras.layers.Dense(1, activation="sigmoid")(bn7)
output = l14 * tf.constant([100.0], dtype=tf.float32)

model = tf.keras.Model(
    inputs={"Image": input_image, "Feature": input_meta}, outputs=output
)



In [6]:
model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Image (InputLayer)  │ (None, 256, 256,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ Image[0][0]       │
│ (MaxPooling2D)      │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 126, 126,  │        224 │ max_pooling2d[0]… │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 63, 63, 8) │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 61, 61,    │      1,168 │ max_pooling2d_1[… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 30, 30,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 28, 28,    │      4,640 │ max_pooling2d_2[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 14, 14,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 12, 12,    │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 9216)      │          0 │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1024)      │  9,438,208 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Feature             │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1036)      │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ Feature[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │    530,944 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 512)       │      2,048 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    262,656 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 512)       │    262,656 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 11,032,049 (42.08 MB)

 Trainable params: 11,026,513 (42.06 MB)

 Non-trainable params: 5,536 (21.62 KB)

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[tf.keras.metrics.RootMeanSquaredError()],
)



In [8]:
BATCH_SIZE = 32

gen_train = train_loader.batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
gen_valid = valid_loader.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

steps_per_epoch = len(train_ids) // BATCH_SIZE
validation_steps = max(1, len(valid_ids) // BATCH_SIZE)

steps_per_epoch, validation_steps



(264, 13)

In [9]:
history = model.fit(
    gen_train,
    steps_per_epoch=steps_per_epoch,
    epochs=2,
    validation_data=gen_valid,
    validation_steps=validation_steps,
)



Epoch 1/2


I0000 00:00:1769082986.693498      62 service.cc:148] XLA service 0x7f8364003010 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769082986.693542      62 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-22 11:56:26.793171: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769082987.260386      62 cuda_dnn.cc:529] Loaded cuDNN version 90300


  3/264 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - loss: 1230.7998 - root_mean_squared_error: 35.0427

I0000 00:00:1769082989.548429      62 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


264/264 ━━━━━━━━━━━━━━━━━━━━ 43s 128ms/step - loss: 703.1886 - root_mean_squared_error: 26.2697 - val_loss: 366.4235 - val_root_mean_squared_error: 19.1422
Epoch 2/2
  4/264 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - loss: 451.4080 - root_mean_squared_error: 21.1607 

2026-01-22 11:57:05.724966: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'loop_add_reduce_subtract_fusion_3', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'loop_add_reduce_subtract_fusion_1', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'loop_add_reduce_subtract_fusion', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'loop_add_reduce_subtract_fusion_5', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'loop_add_reduce_subtract_fusion_4', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'loop_add_reduce_subtract_fusion_6', 4 bytes spill stores, 4 bytes spill loads



264/264 ━━━━━━━━━━━━━━━━━━━━ 29s 101ms/step - loss: 449.2522 - root_mean_squared_error: 21.1883 - val_loss: 349.7286 - val_root_mean_squared_error: 18.7010


In [10]:
test_ids = test_df["Id"].tolist()


def pythonic_loader_kaggle_test():
    for pid in test_ids:
        img = _load_image_np(os.path.join(TEST_IMG_DIR, f"{pid}.jpg"))
        features = test_feat_map[pid]
        yield {"Image": img, "Feature": features}


test_loader = tf.data.Dataset.from_generator(
    pythonic_loader_kaggle_test,
    output_signature={
        "Image": tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
        "Feature": tf.TensorSpec(shape=(12,), dtype=tf.float32),
    },
)



In [11]:
preds = model.predict(test_loader.batch(1), verbose=1)

pawpularity_pred = np.asarray(preds).reshape(-1).astype(np.float32)
pawpularity_pred = np.clip(pawpularity_pred, 0.0, 100.0)

len(pawpularity_pred), len(test_ids)



992/992 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step


2026-01-22 11:57:36.274629: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


(992, 992)

In [12]:
submission = pd.DataFrame({"Id": test_ids, "Pawpularity": pawpularity_pred})
submission.to_csv("submission.csv", index=False)

sample = pd.read_csv(SAMPLE_SUB_PATH)
assert list(submission.columns) == list(sample.columns)
assert submission.shape[0] == sample.shape[0]
submission.head()

,Id,Pawpularity
0,ee51b99832f1ba868f646df93d2b6b81,42.027439
1,caddfb3f8bff9c4b95dbe022018eea21,43.961517
2,582eeabd4a448a53ebb79995888a4b0b,39.558022
3,afc1ad7f0c5eea880759d09e77f7deee,38.404881
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,35.136967
